# IArma

Binary image classifier (firearm or not) built with a small convolutional neural network in Keras.

In [ ]:
# Imports
import numpy as np
import tensorflow as tf
from IPython.display import Image, display
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

## Model

All images are resized to 64x64 pixels with 3 color channels, so the input shape is `(64, 64, 3)`.

Two convolution + max pooling blocks extract features, then `Flatten` turns the feature maps into a vector for the dense layers.

In [ ]:
classifier = Sequential()

# First convolutional block
classifier.add(Conv2D(32, (3, 3), input_shape = (64, 64, 3), activation = 'relu'))
classifier.add(MaxPooling2D(pool_size = (2, 2)))

# Second convolutional block
classifier.add(Conv2D(32, (3, 3), activation = 'relu'))
classifier.add(MaxPooling2D(pool_size = (2, 2)))

classifier.add(Flatten())

A dense ReLU layer followed by a single sigmoid output, which gives the probability of the image belonging to class 1.

In [ ]:
classifier.add(Dense(units = 128, activation = 'relu'))
classifier.add(Dense(units = 1, activation = 'sigmoid'))

Adam optimizer and binary cross-entropy, the usual pair for a sigmoid output with two classes.

In [ ]:
classifier.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

## Data

The images in `dataset_training` are split into 80% for training and 20% for validation. Augmentation (shear, zoom and horizontal flip) is applied to the training part only.

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True,
                                   validation_split = 0.2)

validation_datagen = ImageDataGenerator(rescale = 1./255,
                                        validation_split = 0.2)

training_set = train_datagen.flow_from_directory('dataset_training',
                                                 target_size = (64, 64),
                                                 batch_size = 32,
                                                 class_mode = 'binary',
                                                 subset = 'training',
                                                 seed = 42)

validation_set = validation_datagen.flow_from_directory('dataset_training',
                                                        target_size = (64, 64),
                                                        batch_size = 32,
                                                        class_mode = 'binary',
                                                        subset = 'validation',
                                                        shuffle = False)

print(training_set.class_indices)

## Training

This can take a while, depending on your computer.

In [ ]:
history = classifier.fit(training_set,
                         epochs = 38,
                         validation_data = validation_set)

## Evaluation

Accuracy, precision, recall and F1 on the validation images, which the model did not see during training.

In [ ]:
probabilities = classifier.predict(validation_set).ravel()
predicted = (probabilities >= 0.5).astype(int)
labels = list(validation_set.class_indices.keys())

print(confusion_matrix(validation_set.classes, predicted))
print(classification_report(validation_set.classes, predicted, target_names = labels))

## Testing with new images

The sigmoid output is the probability of class 1. With the folders sorted alphabetically, `guns` is 0 and `random` is 1, so a value below 0.5 means a firearm was detected.

In [ ]:
firearm_class = training_set.class_indices['guns']

for i in range(1, 12):
    path = f'dataset_test/{i}.jpg'
    test_image = tf.keras.utils.load_img(path, target_size = (64, 64))
    test_image = tf.keras.utils.img_to_array(test_image) / 255.
    test_image = np.expand_dims(test_image, axis = 0)

    probability = classifier.predict(test_image, verbose = 0)[0][0]
    predicted_class = int(probability >= 0.5)

    if predicted_class == firearm_class:
        print(f'{path}: there is a firearm in the image.')
    else:
        print(f'{path}: there is no firearm in the image.')
    display(Image(filename = path, width = 200))